# Getting Started with MetaQuantus

This notebook gives the instructions for how to get started with **MetaQuantus**, with methods explained in the paper:

> [**The Meta-Evaluation Problem in Explainable AI: Identifying Reliable Estimators with MetaQuantus**](https://arxiv.org/abs/2302.07265)

We will use the **cMNIST / MNIST dataset** and a **ResNet-9 / LeNet model** for demonstration.  
Make sure to have **GPU enabled** for performance gains.

---

## What does MetaQuantus do?

Explainable AI (XAI) methods produce *explanations* (e.g. saliency maps) that highlight which input features were most important for a model's prediction.  
But **how do we know which XAI method is the most trustworthy?**  

MetaQuantus answers this by *meta-evaluating* XAI quality metrics using two kinds of stress tests:

| Test type | What it checks |
|-----------|----------------|
| **Resilience test** | Does the metric stay stable when tiny, imperceptible noise is added? |
| **Adversary test** | Does the metric change meaningfully when large, destructive noise is added? |

A **good** XAI quality metric should:
- Score **high** on the resilience test (robust to tiny noise)
- Score **high** on the adversary test (sensitive to large noise)

MetaQuantus runs these tests over **model perturbations** (corrupting the model weights) and **input perturbations** (corrupting the input image), giving us four scores per metric per XAI method.


## 0) Installation

First, clone the MetaQuantus repo and install dependencies.
If you are running in **Google Colab**, a GPU runtime is recommended *(Runtime → Change runtime type → GPU)*.

> **Note:** Run the two install cells below **sequentially**, then **restart the runtime** when prompted
> (Runtime → Restart session). After restarting, skip back to Section 1 – do NOT re-run the install cells.

### Why the restart?
MetaQuantus requires `numpy < 2.0`. Colab ships a newer numpy that is binary-incompatible
with this version of pandas. Re-installing both packages together and then restarting the kernel
ensures Python loads the freshly compiled binaries instead of the cached, mismatched ones.


In [ ]:
# Step 1 – clone repo and install MetaQuantus (pins numpy<2.0)
!git clone https://github.com/annahedstroem/MetaQuantus.git 2>/dev/null || echo 'Already cloned'
!pip install -q captum metaquantus


In [ ]:
# Step 2 – reinstall numpy + pandas against the pinned numpy<2.0 to fix binary incompatibility,
# then restart the kernel so Python loads the fresh binaries.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                        'numpy<2.0', 'pandas'])
print('Packages reinstalled. Restarting kernel now...')
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)


## 1) Preliminaries

### 1.0 Imports & device check

We import all required libraries and check whether a GPU is available.
MetaQuantus is built on top of **Quantus** (the XAI evaluation library) and **PyTorch**.


In [ ]:
from IPython.display import clear_output, display
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
import torch
import captum
import quantus
warnings.filterwarnings('ignore', category=UserWarning)
clear_output()

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
try:
    print('GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi
except:
    print('No GPU enabled – running on CPU (may be slow).')


### 1.1 Configure paths

Set the paths to the asset folder that contains model checkpoints and datasets.
The block below auto-detects whether we are running in **Google Colab** or locally.


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    PATH_ASSETS = '/content/MetaQuantus/tests/assets/'
    path = '/content/drive/MyDrive/Projects'
else:
    PATH_ASSETS = '../assets/'   # adjust if running outside the MetaQuantus repo
    path = '../../'

PATH_DATA = PATH_ASSETS + 'data/'
sys.path.append(f'{path}/MetaQuantus')

import metaquantus
from metaquantus import setup_dataset_models, setup_xai_settings, setup_estimators
print('MetaQuantus imported successfully.')


### 1.2 Load data and model

We load the **MNIST** dataset (hand-written digits 0-9) and a pre-trained **LeNet** classifier.
MetaQuantus ships with helper functions that package the dataset, model, and all evaluation hyper-parameters into a single dictionary.

The five sample images will be visualised so we know what we're working with.


In [ ]:
dataset_name = 'MNIST'

dataset_settings, model_name = setup_dataset_models(
    dataset_name=dataset_name, path_assets=PATH_ASSETS, device=device
)
dataset_kwargs = dataset_settings[dataset_name]['estimator_kwargs']

model   = dataset_settings[dataset_name]['models']['LeNet'].eval()
x_batch = dataset_settings[dataset_name]['x_batch']   # shape: (N, C, H, W)
y_batch = dataset_settings[dataset_name]['y_batch']   # ground-truth class labels
s_batch = dataset_settings[dataset_name]['s_batch']   # segmentation masks

# ── Visualise the first 5 samples ──────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
fig.suptitle('Sample MNIST images used for evaluation', fontsize=14, fontweight='bold')
for i in range(5):
    img = np.moveaxis(x_batch[i], 0, -1).reshape(28, 28)
    axes[i].imshow(img, cmap='gray_r')
    axes[i].set_title(f'Label: {y_batch[i]}', fontsize=11)
    axes[i].axis('off')
plt.tight_layout()
plt.show()
print(f'x_batch shape: {x_batch.shape}  |  Model: {model_name}')


### 1.3 Load XAI methods and generate explanations

We compare two XAI methods:
- **LayerGradCam** – gradient-weighted class activation maps from a convolutional layer. Good at coarse spatial localisation.
- **Saliency** – plain input gradients (∂output/∂input). Fast and simple, but can be noisy.

We generate attribution maps for the first 5 images and visualise them side-by-side.


In [ ]:
xai_setting = ['LayerGradCam', 'Saliency']
xai_methods = setup_xai_settings(
    xai_settings=xai_setting,
    gc_layer=dataset_settings[dataset_name]['gc_layers'][model_name],
    img_size=dataset_kwargs['img_size'],
    nr_channels=dataset_kwargs['nr_channels'],
)

# Generate attributions
explanations = {}
for method, kwargs in xai_methods.items():
    m = dataset_settings[dataset_name]['models']['LeNet'].eval().cpu()
    explanations[method] = quantus.explain(
        model=m, inputs=x_batch[:5], targets=y_batch[:5],
        **{'method': method, **kwargs}
    )

# Visualise attributions
n_methods = len(xai_methods)
fig, axes = plt.subplots(n_methods + 1, 5, figsize=(13, 2.5 * (n_methods + 1)))
fig.suptitle('Input images and their XAI attributions', fontsize=14, fontweight='bold')

for i in range(5):
    img = np.moveaxis(x_batch[i], 0, -1).reshape(28, 28)
    axes[0, i].imshow(img, cmap='gray_r')
    axes[0, i].set_title(f'Input  (y={y_batch[i]})', fontsize=9)
    axes[0, i].axis('off')

for row, (method, attr) in enumerate(explanations.items(), start=1):
    for i in range(5):
        a = attr[i].squeeze()
        axes[row, i].imshow(a, cmap='RdBu_r', vmin=-np.abs(a).max(), vmax=np.abs(a).max())
        axes[row, i].axis('off')
    axes[row, 0].set_ylabel(method, fontsize=10, rotation=0, labelpad=80, va='center')

plt.tight_layout()
plt.show()


### 1.4 Load quality estimators (Quantus metrics)

MetaQuantus wraps **Quantus** metrics, organised into five *categories*:

| Category | Metrics included | Best score |
|----------|-----------------|------------|
| **Robustness** | Max-Sensitivity, Local Lipschitz Estimate | lower ↓ |
| **Randomisation** | Random Logit, Model Parameter Randomisation | lower ↓ |
| **Faithfulness** | Faithfulness Correlation, Pixel-Flipping | higher ↑ |
| **Complexity** | Sparseness, Complexity | depends |
| **Localisation** | Pointing-Game, Relevance Mass Accuracy | higher ↑ |

The `score_direction` field tells MetaQuantus whether a **higher** or **lower** raw score is better.


In [ ]:
estimators = setup_estimators(
    features=dataset_kwargs['features'],
    num_classes=dataset_kwargs['num_classes'],
    img_size=dataset_kwargs['img_size'],
    percentage=dataset_kwargs['percentage'],
    patch_size=dataset_kwargs['patch_size'],
    perturb_baseline=dataset_kwargs['perturb_baseline'],
)

# Pretty-print a summary table
rows = []
for cat, metrics in estimators.items():
    for met_name, met_info in metrics.items():
        rows.append({'Category': cat, 'Metric': met_name, 'Score direction': met_info['score_direction']})
df_estimators = pd.DataFrame(rows)
display(df_estimators.to_string(index=False))


---
## 2) Run Meta-Evaluation

### 2.1 Define the test suite

We create **four perturbation tests**.  
Each test modifies either the *model weights* or the *input image* and re-runs the XAI quality metric.  
The resulting distribution of scores is compared to the baseline (un-perturbed) distribution.

| Test name | Perturbation target | Noise std | Type |
|-----------|-------------------|-----------|------|
| Model Resilience | model weights | 0.001 (tiny) | Resilience |
| Model Adversary  | model weights | 2.0 (large) | Adversary |
| Input Resilience | input image   | 0.001 (tiny) | Resilience |
| Input Adversary  | input image   | 5.0 (large)  | Adversary |

> **What to expect:** A reliable metric should be *stable* under the resilience tests and *sensitive* under the adversary tests.


In [ ]:
from metaquantus import ModelPerturbationTest, InputPerturbationTest
from metaquantus import MetaEvaluation, MetaEvaluationBenchmarking

test_suite = {
    'Model Resilience Test': ModelPerturbationTest(
        noise_type='multiplicative', mean=1.0, std=0.001, type='Resilience'
    ),
    'Model Adversary Test': ModelPerturbationTest(
        noise_type='multiplicative', mean=1.0, std=2.0, type='Adversary'
    ),
    'Input Resilience Test': InputPerturbationTest(
        noise=0.001, type='Resilience'
    ),
    'Input Adversary Test': InputPerturbationTest(
        noise=5.0, type='Adversary'
    ),
}
print('Test suite defined with', len(test_suite), 'tests:')
for name in test_suite:
    print(' •', name)


### 2.2 Run MetaQuantus on the Sparseness metric

We run the meta-evaluation on the **Sparseness** metric (*Chalasani et al., 2020*),
which uses the Gini index to measure how concentrated (sparse) the attribution map is.
A higher Gini = more focused explanations.

**Parameters:**
- `iterations = 5` – number of times each perturbation test is repeated (averaging reduces variance)
- `nr_perturbations = 10` – how many perturbed models/inputs are evaluated per iteration

The `MetaEvaluation` object stores per-test IAC (Inter-rater Agreement Coefficient) scores
between 0 and 1, where **1 = perfect agreement with expected ordering**.


In [ ]:
estimator_category = 'Complexity'
estimator_name = 'Sparseness'

iters = 5
K = 10

meta_evaluator = MetaEvaluation(
    test_suite=test_suite,
    xai_methods=xai_methods,
    iterations=iters,
    nr_perturbations=K,
    write_to_file=False,
)

meta_evaluator(
    estimator=estimators[estimator_category][estimator_name]['init'],
    model=dataset_settings[dataset_name]['models'][model_name],
    x_batch=dataset_settings[dataset_name]['x_batch'],
    y_batch=dataset_settings[dataset_name]['y_batch'],
    a_batch=None,
    s_batch=dataset_settings[dataset_name]['s_batch'],
    device=device,
    score_direction=estimators[estimator_category][estimator_name]['score_direction'],
)
print('Meta-evaluation complete.')


### 2.3 Visualise meta-evaluation results

The four IAC scores below quantify how *reliable* the **Sparseness** metric is
under each perturbation condition, for each XAI method.

- **IAC ≈ 1** → the metric behaves exactly as expected (good!)
- **IAC ≈ 0** → the metric fails to distinguish perturbed from clean (bad)
- **IAC < 0** → the metric is *anti-correlated* with expectation (very bad)

The heatmap makes it easy to spot which XAI method x test combination performs best.


In [ ]:
# ── Extract IAC scores from the meta_evaluator results dict ────────────────
# meta_evaluator.results_meta_consistency_scores is structured as:
#   { test_name: { xai_method: float_score, ... }, ... }

results = meta_evaluator.results_meta_consistency_scores  # dict of dicts

test_names  = list(results.keys())
method_names = list(xai_methods.keys())

# Build a 2-D array: rows = tests, cols = XAI methods
score_matrix = np.array(
    [[results[t].get(m, np.nan) for m in method_names] for t in test_names]
)

# ── Print a readable table ──────────────────────────────────────────────────
df_results = pd.DataFrame(score_matrix, index=test_names, columns=method_names)
print(f'\nIAC scores for [{estimator_name}] metric:\n')
print(df_results.round(3).to_string())

# ── Heatmap ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
cax = ax.imshow(score_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
fig.colorbar(cax, ax=ax, label='IAC score (0 = bad, 1 = good)')

ax.set_xticks(range(len(method_names)))
ax.set_xticklabels(method_names, fontsize=11)
ax.set_yticks(range(len(test_names)))
ax.set_yticklabels(test_names, fontsize=10)
ax.set_title(f'MetaQuantus IAC scores – [{estimator_name}] metric', fontsize=13, fontweight='bold')

for i in range(len(test_names)):
    for j in range(len(method_names)):
        val = score_matrix[i, j]
        txt = f'{val:.2f}' if not np.isnan(val) else 'N/A'
        ax.text(j, i, txt, ha='center', va='center', fontsize=12,
                color='black' if 0.3 < val < 0.7 else 'white')

plt.tight_layout()
plt.show()

# ── Bar chart: average IAC per XAI method ──────────────────────────────────
avg_scores = np.nanmean(score_matrix, axis=0)
colors = ['#2ecc71' if s >= 0.5 else '#e74c3c' for s in avg_scores]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(method_names, avg_scores, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylim(0, 1)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Threshold (0.5)')
ax.set_ylabel('Average IAC score', fontsize=12)
ax.set_title(f'Average reliability: [{estimator_name}] across all tests', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
for bar, val in zip(bars, avg_scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

best_method = method_names[int(np.argmax(avg_scores))]
print(f'\n✅ Leading XAI method for [{estimator_name}]: {best_method} (avg IAC = {avg_scores.max():.3f})')


---
## 3) Run Benchmarking

### 3.1 Define a broader set of Localisation metrics

Benchmarking extends the meta-evaluation to **multiple metrics at once**,
letting us rank them and identify which are most reliable overall.

We test four **Localisation** metrics:
- **Pointing-Game** – is the highest-attribution pixel inside the ground-truth segment?
- **Top-K Intersection** – do the top-K attributed pixels overlap with the segment?
- **Relevance Rank Accuracy** – fraction of top-ranked pixels that are inside the segment.
- **Relevance Mass Accuracy** – fraction of total attribution weight inside the segment.


In [ ]:
estimators_localisation = {
    'Localisation': {
        'Pointing-Game': {
            'init': quantus.PointingGame(
                abs=False, normalise=True,
                normalise_func=quantus.normalise_func.normalise_by_max,
                return_aggregate=False, aggregate_func=np.mean,
                disable_warnings=True,
            ),
            'score_direction': 'higher',
        },
        'Top-K Intersection': {
            'init': quantus.TopKIntersection(
                k=10, abs=False, normalise=True,
                normalise_func=quantus.normalise_func.normalise_by_max,
                return_aggregate=False, aggregate_func=np.mean,
                disable_warnings=True,
            ),
            'score_direction': 'higher',
        },
        'Relevance Rank Accuracy': {
            'init': quantus.RelevanceRankAccuracy(
                abs=False, normalise=True,
                normalise_func=quantus.normalise_func.normalise_by_max,
                return_aggregate=False, aggregate_func=np.mean,
                disable_warnings=True,
            ),
            'score_direction': 'higher',
        },
        'Relevance Mass Accuracy': {
            'init': quantus.RelevanceMassAccuracy(
                abs=False, normalise=True,
                normalise_func=quantus.normalise_func.normalise_by_max,
                return_aggregate=False, aggregate_func=np.mean,
                disable_warnings=True,
            ),
            'score_direction': 'higher',
        },
    }
}
print('Localisation estimators configured:', list(estimators_localisation['Localisation'].keys()))


### 3.2 Run the benchmark

This loops over every combination of dataset × model × metric and calls `MetaEvaluation`
using the *same* `meta_evaluator` settings defined above.  
**Note:** this can take several minutes on CPU.


In [ ]:
benchmark = MetaEvaluationBenchmarking(
    master=meta_evaluator,
    estimators=estimators_localisation,
    experimental_settings=dataset_settings,
    device=device,
)()
print('Benchmarking complete.')


### 3.3 Visualise benchmark results

Now we compare **all four Localisation metrics** side-by-side,
for each XAI method, across all four perturbation tests.

The grouped bar chart below shows average IAC scores — higher is better.
Metrics that consistently score above **0.5** (dashed line) can be considered reliable.

The "**Leading metric**" is the one that is best at revealing differences
between XAI methods — i.e., the one that has the highest discriminative power.


In [ ]:
# ── Safely extract benchmark scores ────────────────────────────────────────
# benchmark is a dict: { dataset: { model: { category: { metric: MetaEvaluation } } } }

bench_rows = []
for ds, ds_val in benchmark.items():
    for model_key, model_val in ds_val.items():
        for cat, cat_val in model_val.items():
            for metric_key, me in cat_val.items():
                # me.results_meta_consistency_scores = { test_name: { xai_method: score } }
                for test_name, test_scores in me.results_meta_consistency_scores.items():
                    for xai_m, score in test_scores.items():
                        bench_rows.append({
                            'Dataset': ds, 'Model': model_key,
                            'Metric': metric_key,
                            'Test': test_name, 'XAI Method': xai_m,
                            'IAC': score,
                        })

df_bench = pd.DataFrame(bench_rows)
print('Benchmark result shape:', df_bench.shape)
display(df_bench.head(10))


In [ ]:
# ── Average IAC per (Metric, XAI Method) over all tests ───────────────────
pivot = df_bench.groupby(['Metric', 'XAI Method'])['IAC'].mean().unstack('XAI Method')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(pivot.index))
width = 0.8 / len(pivot.columns)
palette = ['#3498db', '#e67e22', '#9b59b6', '#1abc9c']

for i, (method, col) in enumerate(pivot.items()):
    offset = (i - len(pivot.columns) / 2 + 0.5) * width
    bars = ax.bar(x + offset, col.values, width=width * 0.9,
                  label=method, color=palette[i % len(palette)], alpha=0.88)

ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=20, ha='right', fontsize=10)
ax.set_ylim(0, 1)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1.2, label='Threshold (0.5)')
ax.set_ylabel('Average IAC score', fontsize=12)
ax.set_title('Benchmarking: Localisation metrics × XAI methods', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
plt.tight_layout()
plt.show()

# ── Heatmap: metric vs test ────────────────────────────────────────────────
pivot2 = df_bench.groupby(['Metric', 'Test'])['IAC'].mean().unstack('Test')

fig2, ax2 = plt.subplots(figsize=(10, 4))
cax2 = ax2.imshow(pivot2.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
fig2.colorbar(cax2, ax=ax2, label='IAC (0=bad, 1=good)')
ax2.set_xticks(range(len(pivot2.columns)))
ax2.set_xticklabels(pivot2.columns, rotation=20, ha='right', fontsize=9)
ax2.set_yticks(range(len(pivot2.index)))
ax2.set_yticklabels(pivot2.index, fontsize=10)
ax2.set_title('Metric reliability by perturbation test (avg over XAI methods)', fontsize=12, fontweight='bold')
for i in range(len(pivot2.index)):
    for j in range(len(pivot2.columns)):
        v = pivot2.values[i, j]
        ax2.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=10,
                 color='black' if 0.3 < v < 0.7 else 'white')
plt.tight_layout()
plt.show()


---
## 4) Summary & Interpretation

The cell below automatically identifies which metric and XAI method lead in each category.


In [ ]:
# ── Overall summary ────────────────────────────────────────────────────────
avg_by_metric = df_bench.groupby('Metric')['IAC'].mean().sort_values(ascending=False)
avg_by_xai    = df_bench.groupby('XAI Method')['IAC'].mean().sort_values(ascending=False)
avg_by_test   = df_bench.groupby('Test')['IAC'].mean().sort_values(ascending=False)

print('=' * 60)
print('SUMMARY – MetaQuantus Benchmarking Results')
print('=' * 60)

print('\nMetrics ranked by average IAC (higher = more reliable):')
for rank, (met, sc) in enumerate(avg_by_metric.items(), 1):
    flag = ' ← LEADING' if rank == 1 else ''
    print(f'  {rank}. {met:<30} IAC = {sc:.3f}{flag}')

print('\nXAI methods ranked by average IAC (higher = more faithfully measured):')
for rank, (xai, sc) in enumerate(avg_by_xai.items(), 1):
    flag = ' ← LEADING' if rank == 1 else ''
    print(f'  {rank}. {xai:<30} IAC = {sc:.3f}{flag}')

print('\nPerturbation tests ranked by discriminative power:')
for rank, (test, sc) in enumerate(avg_by_test.items(), 1):
    flag = ' ← most discriminative' if rank == 1 else ''
    print(f'  {rank}. {test:<35} IAC = {sc:.3f}{flag}')

# Final bar chart – metric ranking
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71' if v >= 0.5 else '#e74c3c' for v in avg_by_metric.values]
ax.barh(avg_by_metric.index[::-1], avg_by_metric.values[::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1.2, label='0.5 threshold')
ax.set_xlabel('Average IAC score', fontsize=12)
ax.set_title('Localisation metric ranking\n(higher = more reliable for evaluating XAI)', fontsize=12, fontweight='bold')
for i, (met, sc) in enumerate(avg_by_metric.iloc[::-1].items()):
    ax.text(sc + 0.01, i, f'{sc:.3f}', va='center', fontsize=10)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('\n[RESULT] The leading metric above is the most trustworthy lens for comparing XAI methods on MNIST.')
print('   Use it when you need a single reliable number to rank your explanations.')
